# 06: 역빈도 가중 문장 결합 증강의 한계 분석

기존 `.venv/Scripts/python.exe` 커널을 선택하고 위에서 아래로 실행하세요. 5번 섹션의 현장 선택, `min_weight`, `weight_power`를 바꾸면 그래프와 표가 갱신됩니다. 숫자 입력으로 슬라이더 간격보다 작은 값도 지정할 수 있습니다.

입력은 이 실험의 `data/train.jsonl`뿐입니다. 경로 설정은 프로젝트 루트 기준이며 커널이 실험 디렉터리에서 시작해도 상위의 프로젝트 루트를 찾습니다. GPU, 모델 가중치, 새 그래프 패키지가 필요하지 않으며 별도 결과 파일을 저장하지 않습니다. 6번 섹션에는 위젯 없이 표시되는 정적 그래프와 표도 있습니다.


## 1. 파라미터와 라이브러리


In [1]:
RUNTIME_PARAMS = {
    'prefix_path': 'experiments/06/data',
    'train_file': 'train.jsonl',
}

HYPER_PARAMS = {
    'min_weight': 0.01,
    'weight_power': 0.7,
    'augmentation_probability': 0.5,  # 학습의 결합 확률입니다. 파트너는 한 개입니다.
    'max_work_types': 20,            # 그래프에 표시할 최대 공종 수입니다.
    'max_records': 12,               # 표에 표시할 선택 배수 상위 레코드 수입니다.
}

import json
import math
from collections import Counter, defaultdict
from html import escape
from pathlib import Path

import ipywidgets as widgets
from IPython.display import HTML, display


## 2. 학습 레코드와 현장별 공종 빈도

빈도는 레코드 수나 시공 수량의 합이 아니라 정답 배열에서 공종이 등장한 횟수입니다. 같은 레코드에 같은 공종이 두 번 있으면 두 번 셉니다. 잘못된 입력은 오류로 알립니다.


In [2]:
def runtime_path(field):
    relative = Path(RUNTIME_PARAMS["prefix_path"]) / RUNTIME_PARAMS[field]
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"프로젝트 루트 기준 입력 경로를 확인하세요: {relative}")


def read_records(path):
    records, identifiers = [], set()
    with Path(path).open(encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, 1):
            if not line.strip():
                continue
            record = json.loads(line)
            messages, metadata = record["messages"], record["metadata"]
            if [message["role"] for message in messages] != ["system", "user", "assistant"]:
                raise ValueError(f"메시지 구성 오류: {line_number}")
            identifier = (metadata["site"], metadata["image_id"])
            if identifier in identifiers:
                raise ValueError(f"중복 식별자: {identifier}")
            identifiers.add(identifier)
            items = json.loads(messages[2]["content"])
            if not isinstance(items, list) or not items:
                raise ValueError(f"정답 배열을 확인하세요: {line_number}")
            if any(not isinstance(item, dict) or not isinstance(item.get("workType"), str)
                   or not item["workType"] for item in items):
                raise ValueError(f"정답 공종을 확인하세요: {line_number}")
            records.append({
                "site": metadata["site"],
                "image_id": metadata["image_id"],
                "work_types": Counter(item["workType"] for item in items),
                "text": messages[1]["content"].split("[보드판 텍스트]\n", 1)[-1],
            })
    if not records:
        raise ValueError("학습 데이터가 비어 있습니다.")
    return records


train_path = runtime_path("train_file")
records = read_records(train_path)
site_records = defaultdict(list)
site_counts = defaultdict(Counter)
for record in records:
    site_records[record["site"]].append(record)
    site_counts[record["site"]].update(record["work_types"])
print(f"입력: {train_path}")
print(f"레코드 {len(records):,}건 / 현장 {len(site_records)}개 / GPU 사용 없음")


입력: C:\개인\001_프로젝트\SiteBoard\experiments\06_weighted_concat_refined\data\train.jsonl
레코드 1,568건 / 현장 9개 / GPU 사용 없음


## 3. 학습 코드와 같은 가중치, 정확한 기대 횟수

현장별로 `원래 가중치 = max(공종 빈도 ** -weight_power)`를 계산하고 `max(원래 가중치, min_weight)`를 적용합니다. 여러 공종을 가진 샘플은 가장 희소한 공종의 가중치를 사용하며 포함된 모든 작업이 함께 선택됩니다. 현장 안에서 가중치 합으로 나누므로 다른 현장의 파트너는 선택하지 않습니다. 자기 자신도 파트너 후보에 포함합니다.

각 현장의 한 에포크당 예상 파트너 추출 수는 `현장 레코드 수 × 결합 확률`입니다. 이를 각 레코드의 선택 확률과 곱해 공종 노출의 기대값을 계산합니다. 임의 추출의 오차가 없는 이론값이며 정확도나 실제 토큰 학습량을 예측하지 않습니다. 토큰 상한에 따른 잘림은 반영하지 않습니다.

전체 보기에서도 현장별로 계산한 뒤 합산합니다. 레코드 표의 전체 선택 확률은 `현장 레코드 비율 × 현장 내 선택 확률`이며, 현장별 보기에서는 현장 내 선택 확률입니다.


In [3]:
def calculate_weights(min_weight, weight_power):
    if not (math.isfinite(min_weight) and 0 <= min_weight <= 1
            and math.isfinite(weight_power) and 0 <= weight_power <= 2):
        raise ValueError("min_weight는 0~1, weight_power는 0~2로 지정하세요.")
    weighted = []
    for site, group in site_records.items():
        raw = [max(1.0 / site_counts[site][wt] ** weight_power for wt in record["work_types"])
               for record in group]
        clipped = [max(value, min_weight) for value in raw]
        total = sum(clipped)
        for record, raw_weight, weight in zip(group, raw, clipped):
            weighted.append({
                **record,
                "raw_weight": raw_weight,
                "weight": weight,
                "probability": weight / total,
                "uniform_probability": 1 / len(group),
                "boost": weight / total * len(group),
                "floor_applied": raw_weight < min_weight,
            })
    return weighted


def expected_exposure(weighted, site):
    probability = HYPER_PARAMS["augmentation_probability"]
    if not math.isfinite(probability) or not 0 <= probability <= 1:
        raise ValueError("결합 확률은 0~1이어야 합니다.")
    selected = [record for record in weighted if site is None or record["site"] == site]
    counts, partner = Counter(), defaultdict(float)
    for record in selected:
        counts.update(record["work_types"])
        expected_draws = len(site_records[record["site"]]) * probability * record["probability"]
        for wt, occurrences in record["work_types"].items():
            partner[wt] += expected_draws * occurrences
    rows = []
    for wt, base in counts.items():
        uniform_partner = probability * base
        rows.append({
            "work_type": wt, "base": base,
            "uniform_partner": uniform_partner, "weighted_partner": partner[wt],
            "partner_boost": partner[wt] / uniform_partner if uniform_partner else None,
            "uniform_total": base + uniform_partner, "weighted_total": base + partner[wt],
            "total_boost": (base + partner[wt]) / (base + uniform_partner),
        })
    return selected, rows


## 4. 시각화

파란색은 균등 선택, 주황색은 현재 가중 선택입니다. 공종 그래프는 파트너로 추가되는 횟수를 비교하며 기본 학습분은 아래 표에서 별도로 확인합니다. 공종이 여럿인 파트너는 한 번 뽑혀도 여러 공종을 노출하므로 공종별 노출 횟수의 합이 추출 횟수보다 클 수 있습니다.


In [4]:
def html_table(headers, rows):
    head = "".join(f"<th>{escape(str(value))}</th>" for value in headers)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row) + "</tr>" for row in rows)
    return f'<div style="overflow:auto"><table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>'


def curve_svg(selected, min_weight, weight_power):
    maximum = max(site_counts[record["site"]][wt] for record in selected for wt in record["work_types"])
    maximum = max(2, maximum)
    width, height, left, right, top, bottom = 900, 260, 80, 860, 35, 205
    low = min(0.01, maximum ** -weight_power) / 2
    x = lambda frequency: left + math.log10(frequency) / math.log10(maximum) * (right - left)
    y = lambda weight: bottom - (math.log10(weight) - math.log10(low)) / -math.log10(low) * (bottom - top)
    parts = [f'<svg role="img" aria-label="빈도별 원래 가중치와 하한 적용 가중치" viewBox="0 0 {width} {height}" style="width:100%;max-width:1000px">']
    for frequency in sorted({1, maximum, *[10 ** i for i in range(1, 6) if 10 ** i < maximum]}):
        px = x(frequency)
        parts.append(f'<line x1="{px}" y1="{top}" x2="{px}" y2="{bottom}" stroke="#e2e8f0"/><text x="{px}" y="225" text-anchor="middle">{frequency}</text>')
    for weight in [10 ** -i for i in range(7) if 10 ** -i >= low]:
        py = y(weight)
        parts.append(f'<line x1="{left}" y1="{py}" x2="{right}" y2="{py}" stroke="#e2e8f0"/><text x="70" y="{py + 4}" text-anchor="end">{weight:g}</text>')
    frequencies = [maximum ** (i / 120) for i in range(121)]
    for color, clipped in [("#2563eb", False), ("#ea580c", True)]:
        points = []
        for frequency in frequencies:
            weight = frequency ** -weight_power
            if clipped:
                weight = max(weight, min_weight)
            points.append(f"{x(frequency):.2f},{y(weight):.2f}")
        dash = ' stroke-dasharray="6 4"' if not clipped else ''
        parts.append(f'<polyline points="{" ".join(points)}" fill="none" stroke="{color}" stroke-width="3"{dash}/>')
    parts.append('<text x="80" y="20">정규화 전 가중치: 파란 점선 = 역빈도, 주황 실선 = 하한 적용</text><text x="450" y="250" text-anchor="middle">현장 안의 공종 등장 빈도 (양 축 로그 눈금)</text></svg>')
    return "".join(parts)


def bars_svg(rows):
    width, left, right = 1000, 235, 895
    maximum = max(max(row["uniform_partner"], row["weighted_partner"]) for row in rows) or 1
    height = 75 + 48 * len(rows)
    parts = [f'<svg role="img" aria-label="공종별 균등 및 가중 파트너 예상 노출 횟수" viewBox="0 0 {width} {height}" style="width:100%;max-width:1100px">']
    for fraction in [0, 0.25, 0.5, 0.75, 1]:
        px = left + fraction * (right - left)
        parts.append(f'<line x1="{px}" y1="25" x2="{px}" y2="{height - 25}" stroke="#e2e8f0"/><text x="{px}" y="18" text-anchor="middle">{maximum * fraction:.1f}</text>')
    for index, row in enumerate(rows):
        py = 44 + 48 * index
        parts.append(f'<text x="{left - 12}" y="{py + 11}" text-anchor="end">{escape(row["work_type"])}</text>')
        for shift, key, color in [(0, "uniform_partner", "#2563eb"), (17, "weighted_partner", "#ea580c")]:
            value = row[key]
            length = value / maximum * (right - left)
            parts.append(f'<rect x="{left}" y="{py + shift}" width="{length}" height="13" fill="{color}" rx="2"/><text x="{left + length + 6}" y="{py + shift + 11}">{value:.2f}</text>')
    parts.append(f'<text x="{left}" y="{height - 5}">파트너로 추가되는 공종 등장 횟수 / 에포크: 파랑 = 균등, 주황 = 가중</text></svg>')
    return "".join(parts)


def render_view(site, min_weight, weight_power, order):
    selected, exposure = expected_exposure(calculate_weights(min_weight, weight_power), site)
    if order == "boost":
        exposure.sort(key=lambda row: (-(row["partner_boost"] or 0), row["work_type"]))
    else:
        exposure.sort(key=lambda row: (row["base"] if order == "rare" else -row["base"], row["work_type"]))
    shown = exposure[:HYPER_PARAMS["max_work_types"]]
    floor_count = sum(record["floor_applied"] for record in selected)
    maximum_boost = max(record["boost"] for record in selected)
    probability = HYPER_PARAMS["augmentation_probability"]
    cards = [("레코드", f"{len(selected):,}건"), ("하한에 의해 상승", f"{floor_count:,}건"),
             ("레코드 최대 선택 배수", f"{maximum_boost:.2f}배"), ("예상 파트너 추출", f"{len(selected) * probability:,.1f}회/에포크")]
    style = """<style>.weight-view{font-family:'Malgun Gothic','Apple SD Gothic Neo',sans-serif;color:#1e293b;background:#fff;padding:18px;border:1px solid #e2e8f0;border-radius:12px}.weight-view svg{font-family:inherit;font-size:12px;color:#1e293b}.weight-view svg text{fill:#334155}.weight-view table{border-collapse:collapse;font-size:12px;white-space:nowrap;margin:12px 0}.weight-view th,.weight-view td{padding:7px 10px;border-bottom:1px solid #e2e8f0;text-align:right}.weight-view th{background:#f1f5f9}.weight-view td:first-child,.weight-view th:first-child{text-align:left}.weight-view h3{margin:22px 0 10px;font-size:16px}.weight-view p{font-size:13px;line-height:1.7}</style>"""
    cards_html = ''.join(f'<div style="padding:12px 16px;background:#f1f5f9;border-radius:8px"><div style="font-size:12px">{escape(title)}</div><strong style="font-size:20px">{escape(value)}</strong></div>' for title, value in cards)
    label = site or "전체 현장"
    note = ("현재 선택 범위에서는 모두 균등 선택입니다." if all(abs(record["boost"] - 1) < 1e-10 for record in selected)
            else "희소 공종을 포함한 레코드의 선택 확률이 달라집니다. 같은 레코드의 다른 공종도 함께 선택됩니다.")
    if floor_count == 0:
        note += " 하한에 걸린 레코드가 없어 min_weight를 더 낮춰도 선택 확률은 같습니다."
    table = html_table(["공종", "원본 등장", "균등 파트너", "가중 파트너", "파트너 배수", "균등 학습 합계", "가중 학습 합계", "합계 배수"],
        [(row["work_type"], row["base"], f'{row["uniform_partner"]:.2f}', f'{row["weighted_partner"]:.2f}',
          f'{row["partner_boost"]:.2f}' if row["partner_boost"] is not None else "해당 없음",
          f'{row["uniform_total"]:.2f}', f'{row["weighted_total"]:.2f}', f'{row["total_boost"]:.2f}') for row in exposure])
    top_records = sorted(selected, key=lambda record: (-record["boost"], record["site"], record["image_id"]))[:HYPER_PARAMS["max_records"]]
    record_rows = []
    for record in top_records:
        mass = len(site_records[record["site"]]) / len(records) if site is None else 1
        record_rows.append((record["site"], record["image_id"],
            ", ".join(f"{wt}({count})" for wt, count in record["work_types"].items()),
            f'{record["raw_weight"]:.6f}', f'{record["weight"]:.6f}',
            f'{record["probability"] * mass * 100:.4f}%', f'{record["boost"]:.2f}', record["text"]))
    record_table = html_table(["현장", "사진", "공종(배열 등장 횟수)", "역빈도", "하한 적용", "파트너 선택 확률", "균등 대비 배수", "입력 텍스트"], record_rows)
    return (style + f'<div class="weight-view"><h3>{escape(label)}: min_weight={min_weight:g}, weight_power={weight_power:g}</h3>'
            + '<div style="display:flex;flex-wrap:wrap;gap:12px">' + cards_html + '</div>'
            + f'<p>{escape(note)}</p><h3>1. 빈도에서 가중치로</h3>' + curve_svg(selected, min_weight, weight_power)
            + f'<h3>2. 공종별 파트너 노출: {len(shown)}/{len(exposure)}개 공종 표시</h3>' + bars_svg(shown)
            + '<h3>3. 기본 학습분을 포함한 공종 노출</h3><p>학습 합계 = 원본 등장 + 예상 파트너 등장. 가중치가 적용되는 부분은 파트너뿐입니다. 표는 모든 공종을 표시합니다.</p>' + table
            + '<h3>4. 선택 배수 상위 레코드</h3><p>전체 보기에서는 현장 선택 비율을 반영한 확률입니다. 서로 다른 현장 사이에서 공종 빈도를 직접 비교해 파트너를 선택하지 않습니다.</p>' + record_table + '</div>')


## 5. 파라미터를 바꿔 확인하기

처음에는 `min_weight=1`과 `0.01`을 비교하고, 이어 `0`으로 바꿔 하한 제한이 남아 있는지 확인하세요. `weight_power=0`도 균등 선택입니다. 슬라이더는 마우스를 놓으면 반영되고 숫자 입력은 Enter 또는 입력칸 밖 클릭으로 반영됩니다.

그래프와 표는 아래 HTML 위젯의 내용이 바뀌면서 갱신됩니다. 위젯 내용이 보이지 않는 편집기에서는 6번 섹션의 일반 출력 셀을 사용하세요. 변경은 이 노트북의 계산에만 반영되며 학습 노트북 설정은 수정하지 않습니다.


In [5]:
def show_weights(min_weight=None, weight_power=None, site=None, order="rare"):
    min_weight = HYPER_PARAMS["min_weight"] if min_weight is None else min_weight
    weight_power = HYPER_PARAMS["weight_power"] if weight_power is None else weight_power
    display(HTML(render_view(site, min_weight, weight_power, order)))


site_control = widgets.Dropdown(options=[("전체 현장", None)] + [(site, site) for site in sorted(site_records)], description="현장")
min_control = widgets.FloatSlider(value=HYPER_PARAMS["min_weight"], min=0, max=1, step=0.001,
                                  description="min_weight", readout=False, continuous_update=False,
                                  style={"description_width": "110px"}, layout=widgets.Layout(width="450px"))
min_number = widgets.BoundedFloatText(value=min_control.value, min=0, max=1, step=0.001, layout=widgets.Layout(width="130px"))
power_control = widgets.FloatSlider(value=HYPER_PARAMS["weight_power"], min=0, max=2, step=0.05,
                                    description="weight_power", readout=False, continuous_update=False,
                                    style={"description_width": "110px"}, layout=widgets.Layout(width="450px"))
power_number = widgets.BoundedFloatText(value=power_control.value, min=0, max=2, step=0.05, layout=widgets.Layout(width="130px"))
links = [widgets.link((min_control, "value"), (min_number, "value")),
         widgets.link((power_control, "value"), (power_number, "value"))]
order_control = widgets.Dropdown(options=[("빈도 낮은 순", "rare"), ("보정 효과 큰 순", "boost"), ("빈도 높은 순", "frequent")], description="공종 정렬")
buttons = []
for value in (1.0, 0.1, 0.01, 0.001, 0.0):
    button = widgets.Button(description=f"min={value:g}", layout=widgets.Layout(width="105px"))
    button.on_click(lambda _, value=value: setattr(min_control, "value", value))
    buttons.append(button)
output = widgets.HTML(layout=widgets.Layout(width="100%"))


def refresh(change=None):
    output.value = render_view(site_control.value, min_control.value, power_control.value, order_control.value)


for control in (site_control, min_control, power_control, order_control):
    control.observe(refresh, names="value")
display(widgets.VBox([widgets.HBox([site_control, order_control]),
                      widgets.HBox([min_control, min_number]), widgets.HBox([power_control, power_number]),
                      widgets.HBox(buttons), output]))
refresh()


## 6. 일반 셀 출력으로 보기

아래 셀은 HTML 위젯 없이 그래프와 표를 일반 셀 출력에 표시합니다. 위에서 숫자를 바꾼 뒤 이 셀만 다시 실행하면 현재 제어값을 반영합니다. 처음에는 기본 설정의 미리보기가 저장되어 있습니다.

위젯 대신 코드에서 직접 지정하려면 `show_weights(min_weight=0.01, weight_power=0.7, site="GJ")`처럼 호출하세요. 전체 현장은 `site=None`입니다.


In [6]:
show_weights(min_control.value, power_control.value, site_control.value, order_control.value)


공종,원본 등장,균등 파트너,가중 파트너,파트너 배수,균등 학습 합계,가중 학습 합계,합계 배수
무보온입상차열재마감,6,3.00,25.31,8.44,9.00,31.31,3.48
보온입상차열재마감,9,4.50,5.55,1.23,13.50,14.55,1.08
PVC입상,10,5.00,21.96,4.39,15.00,31.96,2.13
보온덕트입상,10,5.00,11.00,2.20,15.00,21.00,1.40
무보온덕트입상,11,5.50,36.09,6.56,16.50,47.09,2.85
오픈구,21,10.50,58.19,5.54,31.50,79.19,2.51
댐퍼팽창테이프,26,13.00,29.26,2.25,39.00,55.26,1.42
덕트마감,27,13.50,48.85,3.62,40.50,75.85,1.87
실란트마감벽체,68,34.00,48.95,1.44,102.00,116.95,1.15
금속관입상,108,54.00,109.53,2.03,162.00,217.53,1.34
